In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

In [1]:
import sys
import os
import seaborn as sns
import numpy as np

module_path = os.path.abspath(os.path.join('..', ''))
if module_path not in sys.path:
    sys.path.append(module_path)

from rect_prune_ood.lib.util.paths import get_top_level_path

# This enables imports like
# from rect_prune_ood.lib.prunging import X

[0. 0. 0. 1. 0. 1. 0. 0. 0. 1.]


In [4]:
results_dir = os.path.join(get_top_level_path(), 'results', 'imagenet', "baseline_no_pruning")

activations = np.load(os.path.join(results_dir, "imagenet_activations.npy"), allow_pickle=True)

activations.shape

activations_perturbed = np.load(os.path.join(results_dir, "imagenet_perturbed_activations.npy"), allow_pickle=True)

print(activations_perturbed.shape)

all_activations = result = np.concatenate((activations, activations_perturbed), axis=0)

(200000, 2048)


In [7]:
def imagewise_variance(normal_activations, perturbed_activations, p=90, num_augments=4):
    assert perturbed_activations.shape[0] % num_augments == 0
    #group perturbed activations by image
    grouped_perturbed = perturbed_activations.reshape(normal_activations.shape[0], num_augments, normal_activations.shape[1])
    #take mean of imagewise activations to get rough activation level on perturbed images in general
    #maybe this step can be skipped for experimentiation
    mean_imagewise_perturbed = grouped_perturbed.mean(axis=1)

    #concatenate perturbed and normal acticvations
    all_activations = np.stack((normal_activations, mean_imagewise_perturbed), axis=1)

    iamgewise_variance = np.var(all_activations, axis=1)

    max_indices = np.argmax(iamgewise_variance, axis=1)

    index_counts = np.bincount(max_indices, minlength=2048)
    ranked_indices = np.argsort(index_counts)[::-1]

    top_10th_count = int(len(ranked_indices) * 0.1)
    ranked_indices = ranked_indices[top_10th_count:]

    return ranked_indices


In [8]:
imagewise_variance(activations, activations_perturbed)

array([   0,    0,    0, ...,  541, 1688, 1690])

In [6]:
all_activations.shape

results_dir2 = os.path.join(get_top_level_path(), 'results', 'masks')
mask = np.load(os.path.join(results_dir2, "prune_top_10_id_idperturbed_variance.npy"), allow_pickle=True)

variance = np.std(all_activations, axis=0)

# Calculate the 10th percentile
percentile_90 = np.percentile(variance, 90)

# Get the indices of values in the top 10%
top_10_percent_indices = np.where(variance <= percentile_90)[0]

assert mask.all() == top_10_percent_indices.all()

In [14]:
#excluding top 10 highes activation variances

variance = np.std(all_activations, axis=0)

# Calculate the 10th percentile
percentile_90 = np.percentile(variance, 90)

# Get the indices of values in the top 10%
top_10_percent_indices = np.where(variance <= percentile_90)[0]

print(top_10_percent_indices)

print((2048*2047/2), top_10_percent_indices.sum())

np.save("prune_top_10_id_idperturbed_variance.npy", top_10_percent_indices)

[   0    1    2 ... 2045 2046 2047]
2096128.0 1890338


In [10]:
variance = np.std(activations, axis=0)
variance_percentile = np.percentile(variance, 90)

print(variance_percentile)
print(variance.shape)
print(variance.mean())
print(variance.max(), variance.min())

0.5605087
(2048,)
0.44736767
1.1170037 0.112264246


In [ ]:
import matplotlib.pyplot as plt

# Create the bar plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(variance)), variance, width=1.0, color='blue', edgecolor='none')
plt.xlabel("Index")
plt.ylabel("Value")
plt.title("Bar Plot of 2048 Values")
plt.show()